# **Data Analytics in Commerce**

In [ ]:
# Uploading libraries

library(tidyverse)
library(lubridate)
library(ggplot2)

In [ ]:
# loading Data
orders <- read.csv('/content/olist_orders_dataset.csv')
items <- read.csv('/content/olist_order_items_dataset.csv')
products <- read.csv('/content/olist_products_dataset.csv')
customers <- read.csv('/content/olist_customers_dataset.csv')
reviews <- read.csv('/content/olist_order_reviews_dataset.csv')


In [ ]:
#cleaning reviews
reviews_clean <- reviews %>%
  distinct(order_id, .keep_all = TRUE)

reviews_clean$review_comment_title[is.na(reviews_clean$review_comment_title)] <- "No comment"
reviews_clean$review_comment_message[is.na(reviews_clean$review_comment_message)] <- "No comment"

In [ ]:
#Merging the dataset

df <- orders %>%
  inner_join(items, by = "order_id") %>%
  inner_join(products, by = "product_id") %>%
  inner_join(customers, by = "customer_id") %>%
  inner_join(reviews_clean, by = "order_id")

In [ ]:
#checking data structure
head(orders)
str(orders)

head(items)
str(items)

head(products)
str(products)

head(customers)
str(customers)

head(reviews)
str(reviews)

In [ ]:
#check summary
summary(orders)
summary(items)
summary(products)
summary(customers)
summary(reviews)

In [ ]:
#check missing values

colSums(is.na(orders))
colSums(is.na(items))
colSums(is.na(products))
colSums(is.na(customers))
colSums(is.na(reviews))

In [ ]:
# Replacing  missing value with median

df$product_name_lenght[is.na(df$product_name_lenght)] <- median(df$product_name_lenght, na.rm = TRUE)

df$product_description_lenght[is.na(df$product_description_lenght)] <- median(df$product_description_lenght, na.rm = TRUE)

df$product_photos_qty[is.na(df$product_photos_qty)] <- median(df$product_photos_qty, na.rm = TRUE)

df$product_weight_g[is.na(df$product_weight_g)] <- median(df$product_weight_g, na.rm = TRUE)

df$product_length_cm[is.na(df$product_length_cm)] <- median(df$product_length_cm, na.rm = TRUE)

df$product_height_cm[is.na(df$product_height_cm)] <- median(df$product_height_cm, na.rm = TRUE)

df$product_width_cm[is.na(df$product_width_cm)] <- median(df$product_width_cm, na.rm = TRUE)

In [ ]:
#convert
df$order_purchase_timestamp <- ymd_hms(df$order_purchase_timestamp)
df$order_delivered_customer_date <- ymd_hms(df$order_delivered_customer_date)
df$order_estimated_delivery_date <- ymd_hms(df$order_estimated_delivery_date)

In [ ]:
#checking
colSums(is.na(df))

In [ ]:
#Checking duplicate in order id
df %>%
  count(order_id) %>%
  filter(n > 1)

In [ ]:
# Check duplicate order_id in reviews
reviews %>%
  count(order_id) %>%
  filter(n > 1)

In [ ]:
#How many duplicated IDs exist
reviews %>%
  count(order_id) %>%
  filter(n > 1) %>%
  nrow()

In [ ]:
#Removing duplicates
df <- distinct(df)

In [ ]:
# Keeping only one review per order
reviews_clean <- reviews %>%
  distinct(order_id, .keep_all = TRUE)

In [ ]:
#check duplicates
sum(duplicated(df))

In [ ]:
#Feature engineering

df <- df %>%
  mutate(
    delivery_delay = as.numeric(order_delivered_customer_date - order_estimated_delivery_date),

    delivery_status = case_when(
      delivery_delay > 0 ~ "Late",
      delivery_delay < 0 ~ "Early",
      TRUE ~ "On Time"
    )
  )

In [ ]:
summary(df$delivery_delay)
table(df$delivery_status)

In [ ]:
df <- df %>%
  filter(!is.na(order_delivered_customer_date))

In [ ]:
df <- df %>%
  filter(!is.na(order_estimated_delivery_date))

In [ ]:
df <- df %>%
  mutate(
    delivery_delay = as.numeric(order_delivered_customer_date - order_estimated_delivery_date)
  )

In [ ]:
df<- df %>%
  mutate(delivery_delay_days = delivery_delay / 86400)

In [ ]:
sum(is.na(df$delivery_delay))

In [ ]:
write.csv(df, "clean_datas.csv", row.names = FALSE)

# **VISUALS**

In [ ]:
# Visual of orders over time
df%>%
  mutate(order_date = as.Date(order_purchase_timestamp)) %>%
  count(order_date) %>%
  ggplot(aes(x = order_date, y = n)) +
  geom_line() +
  labs(title = "Orders Over Time", x = "Date", y = "Orders")

In [ ]:
#Top Categories

df %>%
  count(product_category_name, sort = TRUE) %>%
  slice_head(n = 10) %>%
  ggplot(aes(x = reorder(product_category_name, n), y = n)) +
  geom_col() +
  coord_flip() +
  labs(title = "Top 10 Categories", x = "Category", y = "Orders")

In [ ]:
#Delivery Delay Histogram

df %>%
  filter(!is.na(delivery_delay_days),
         delivery_delay_days > -20,
         delivery_delay_days < 20) %>%
  ggplot(aes(x = delivery_delay_days)) +
  geom_histogram(bins = 50) +
  labs(title = "Delivery Delay (Days)", x = "Days", y = "Count")

In [ ]:
#Review Scores
df%>%
  ggplot(aes(x = factor(review_score))) +
  geom_bar() +
  labs(title = "Review Scores", x = "Score", y = "Count")

In [ ]:
#Orders by State

df %>%
  count(customer_state, sort = TRUE) %>%
  ggplot(aes(x = reorder(customer_state, n), y = n)) +
  geom_col() +
  coord_flip() +
  labs(title = "Orders by State", x = "State", y = "Orders")

In [ ]:
cat("==== SUMMARY ====\n")

cat("Total Orders:", nrow(df), "\n")

cat("Average Delivery Delay (days):",
    mean(df$delivery_delay_days, na.rm = TRUE), "\n")

cat("Average Review Score:",
    mean(df$review_score, na.rm = TRUE), "\n")

cat("Top State:",
    df%>% count(customer_state, sort = TRUE) %>% slice(1) %>% pull(customer_state),
    "\n")

cat("Top Category:",
    df%>% count(product_category_name, sort = TRUE) %>% slice(1) %>% pull(product_category_name),
    "\n")